In [1]:
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

## Считаем доход от каждой поездки

In [2]:
after_2020 = pd.read_parquet('..//data//after_2020.parquet')

In [3]:
prices = pd.read_csv('..//data//prices.csv')
prices.columns = prices.columns.str.lower()
prices = prices[prices['year'] >= 2020]

prices = prices[(prices['tariff_type'].isin(['Single Ride', 'Annual Pass']))]

In [4]:
prices['bike_type'] = prices['bike_type'].map({'Classic Bike': 'classic_bike',
                                               'Classic Bikes': 'classic_bike',
                                                'Ebike': 'electric_bike',
                                                'E-Bikes': 'electric_bike',
                                                'Scooters': 'electric_scooter'
                                               })

prices['tariff_type'] = prices['tariff_type'].map({'Single Ride': 'casual',
                                                    'Annual Pass': 'member',
                                                })

In [5]:
tariff_lookup = {
    (row['year'], row['tariff_type'], row['bike_type']): (
        row['unlock_fee'],
        row['free_period'],
        row['per_minute_rate']
    )
    for _, row in prices.iterrows()
}

In [6]:
after_2020['year'] = after_2020['started_at'].dt.year

In [7]:
nulls = 0
nulls_counts = {2020: 0, 2021: 0, 2022: 0, 2023: 0, 2024: 0}

def count_ride_income_fast(x):
    global nulls
    year = x['year']
    user_type = x['user_type']
    rideable_type = x['rideable_type']
    duration = x['trip_duration'] // 60 + 1

    if rideable_type == 'docked_bike':
        rideable_type = 'classic_bike'
    elif rideable_type == 'electric_scooter':
        rideable_type = 'electric_bike'

    key = (year, user_type, rideable_type)
    if key not in tariff_lookup.keys():
        nulls += 1
        nulls_counts[year] += 1

    unlock_fee, free_minutes, rate = tariff_lookup.get(
        key, (1, 45, 0.18)
    )

    income = unlock_fee + max(duration - free_minutes, 0) * rate
    return income


after_2020['income'] = after_2020.progress_apply(count_ride_income_fast, axis=1)

100%|██████████| 24454526/24454526 [03:34<00:00, 113844.20it/s]


In [8]:
after_2020.to_parquet('..//data//after_2020.parquet', index=False)